# Ogham OCR: Colab Pro Training

Fine-tune TrOCR for ancient Ogham inscription recognition using 200k synthetic images.

**What this notebook does:**
1. **Setup** -- Install deps, mount Drive, verify data
2. **Generate dataset** -- 200k sharded JPEG images (if not already on Drive)
3. **Phase 1** -- Synthetic-only training (compare Ogham Unicode vs Latin transliteration)
4. **Phase 2** -- Curriculum learning mixing real stone data with synthetic
5. **Evaluate** -- CER metrics, sample predictions, loss curves

**Requirements:** Colab Pro with A100 GPU, ~16GB Drive space.

Each cell is independently runnable after a session disconnect (checkpoint resume).

---
## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q transformers accelerate Pillow editdistance tqdm albumentations opencv-python-headless

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create project structure on Drive
import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
for subdir in ['datasets', 'checkpoints', 'logs', 'experiments']:
    os.makedirs(f'{DRIVE_ROOT}/{subdir}', exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
# Clone repository (re-run safe -- pulls latest if already cloned)
import os
REPO_DIR = '/content/ogham_ocr'

if os.path.exists(REPO_DIR):
    print("Repository already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}

%cd {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Verify fonts are available
# If fonts are not on Drive yet, upload them manually to:
#   /content/drive/MyDrive/ogham_ocr/datasets/fonts/
# Required: at least 1 Ogham-compatible .ttf or .otf font

from pathlib import Path

FONT_DIR = Path(f'{DRIVE_ROOT}/datasets/fonts')
LOCAL_FONT_DIR = Path(f'{REPO_DIR}/data/fonts')

# Ensure font directory exists on Drive before attempting copy
FONT_DIR.mkdir(parents=True, exist_ok=True)

# Try Drive first, fall back to repo
font_files = list(FONT_DIR.glob('*.ttf')) + list(FONT_DIR.glob('*.otf'))
if not font_files:
    font_files = list(LOCAL_FONT_DIR.glob('*.ttf')) + list(LOCAL_FONT_DIR.glob('*.otf'))
    if font_files:
        # Copy repo fonts to Drive for persistence
        import shutil
        for f in font_files:
            shutil.copy2(f, FONT_DIR / f.name)
        print(f"Copied {len(font_files)} fonts to Drive")
    else:
        print("WARNING: No Ogham fonts found!")
        print(f"  Upload .ttf/.otf files to: {FONT_DIR}")
        print("  Recommended: BabelStone Ogham, Noto Sans Ogham")

font_files = list(FONT_DIR.glob('*.ttf')) + list(FONT_DIR.glob('*.otf'))
print(f"Fonts available: {len(font_files)}")
for f in font_files:
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

---
## 2. Configuration

Edit the cell below to configure your experiment. These values persist across cells.

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION -- edit these values
# ============================================================

# Model
MODEL_SIZE = 'small'            # 'small' (62M params) or 'base' (334M params)
INIT_STRATEGY = 'latin'         # 'latin' (best), 'random', or 'zero'

# Dataset generation
N_TRAIN_SAMPLES = 200_000       # Total synthetic training images
N_VAL_SAMPLES = 5_000           # Synthetic validation images
N_SHARDS = 10                   # Number of shards for training data
GEN_WORKERS = 4                 # Parallel workers for generation
IMAGE_FORMAT = 'jpeg'           # 'jpeg' (~8GB total) or 'png' (~57GB total)

# Phase 1: Synthetic-only training
P1_EPOCHS = 20                  # Epochs for Phase 1
P1_BATCH_SIZE = 16              # A100: 16-32, T4: 4-8
P1_LR = 5e-5                    # Learning rate
P1_FREEZE_ENCODER = 5           # Freeze encoder for N epochs
P1_MODE = 'compare'             # 'ogham', 'latin', or 'compare' (runs both)

# Phase 2: Curriculum with real data
P2_EPOCHS = 50                  # Epochs for Phase 2
P2_BATCH_SIZE = 16
P2_LR = 2e-5                    # Lower LR for fine-tuning

# Storage strategy:
# - Synthetic data → Colab local disk (fast, ~100GB free, regenerated each session)
# - Checkpoints + results → Drive (persistent, small ~500MB)
# - Real data → Drive (persistent, small ~200MB)
LOCAL_ROOT = '/content/ogham_data'
os.makedirs(LOCAL_ROOT, exist_ok=True)

SYNTH_DATA_DIR = f'{LOCAL_ROOT}/synthetic_200k'
VAL_DATA_DIR = f'{LOCAL_ROOT}/synthetic_val'
REAL_DATA_DIR = f'{DRIVE_ROOT}/datasets/real'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
FONT_DIR_STR = str(FONT_DIR)

# ============================================================
print("Configuration:")
print(f"  Model: trocr-{MODEL_SIZE}-stage1 ({INIT_STRATEGY} init)")
print(f"  Training data: {N_TRAIN_SAMPLES:,} images in {N_SHARDS} shards")
print(f"  Validation data: {N_VAL_SAMPLES:,} images")
print(f"  Phase 1: {P1_EPOCHS} epochs, batch {P1_BATCH_SIZE}, mode={P1_MODE}")
print(f"  Phase 2: {P2_EPOCHS} epochs, batch {P2_BATCH_SIZE}")
est_size_gb = N_TRAIN_SAMPLES * (40 if IMAGE_FORMAT == 'jpeg' else 290) / 1e6
print(f"  Est. dataset size: ~{est_size_gb:.1f} GB ({IMAGE_FORMAT.upper()})")
print(f"\n  Synthetic data: {LOCAL_ROOT} (local, regenerated each session)")
print(f"  Checkpoints:    {CHECKPOINT_DIR} (Drive, persistent)")
print(f"  Real data:      {REAL_DATA_DIR} (Drive, persistent)")

In [ ]:
# Free Drive space: remove synthetic data that now lives on local disk
# This cell is safe to re-run. Only deletes synthetic datasets, not checkpoints.

import shutil
from pathlib import Path

old_synth = Path(f'{DRIVE_ROOT}/datasets/synthetic_200k')
old_val = Path(f'{DRIVE_ROOT}/datasets/synthetic_val')
freed = 0

for d in [old_synth, old_val]:
    if d.exists():
        import subprocess
        result = subprocess.run(['du', '-sb', str(d)], capture_output=True, text=True)
        size = int(result.stdout.split()[0]) if result.stdout.strip() else 0
        freed += size
        shutil.rmtree(d)
        print(f"Removed {d.name} ({size / 1e9:.1f} GB)")

# Also clean up redundant final checkpoints (best_* is kept)
ckpt_dir = Path(CHECKPOINT_DIR)
for final in ckpt_dir.glob('final_*'):
    if final.is_dir():
        shutil.rmtree(final)
        print(f"Removed redundant {final.name}")

if freed > 0:
    print(f"\nFreed ~{freed / 1e9:.1f} GB of Drive space")
else:
    print("No old synthetic data on Drive to clean up.")

---
## 3. Generate Dataset

Generate 200k synthetic training images + 5k validation images.
Stored on Drive so you only need to do this once.

**Time estimate:** ~17 min with 4 workers for 200k JPEG images.

In [ ]:
# Check if dataset already exists on Drive
from pathlib import Path
import json

synth_path = Path(SYNTH_DATA_DIR)
val_path = Path(VAL_DATA_DIR)

synth_exists = False
config_file = synth_path / 'generation_config.json'
if config_file.exists():
    with open(config_file) as f:
        gen_config = json.load(f)
    existing_n = gen_config.get('total_images', 0)
    synth_exists = existing_n >= N_TRAIN_SAMPLES
    if synth_exists:
        print(f"Training dataset already exists: {existing_n:,} images in {synth_path}")
    else:
        print(f"Existing dataset has {existing_n:,} images, need {N_TRAIN_SAMPLES:,}")

val_exists = (val_path / 'labels.csv').exists()
if val_exists:
    import csv
    with open(val_path / 'labels.csv') as f:
        n_val = sum(1 for _ in csv.reader(f)) - 1  # minus header
    print(f"Validation dataset already exists: {n_val:,} images in {val_path}")

if synth_exists and val_exists:
    print("\nAll datasets ready! Skip to Phase 1.")
else:
    print("\nNeed to generate dataset(s). Run the next cell.")

In [ ]:
# Generate training dataset (skip if already exists)
import time

if not synth_exists:
    print(f"Generating {N_TRAIN_SAMPLES:,} training images in {N_SHARDS} shards...")
    start = time.time()

    !python scripts/generate_demo_dataset.py \
        --n {N_TRAIN_SAMPLES} \
        --format {IMAGE_FORMAT} \
        --shards {N_SHARDS} \
        --workers {GEN_WORKERS} \
        --difficulty mixed \
        --realism medium \
        --seed 42 \
        --output-dir {SYNTH_DATA_DIR}

    elapsed = time.time() - start
    print(f"\nGeneration complete in {elapsed/60:.1f} minutes")
else:
    print("Training dataset already exists, skipping generation.")

# Generate validation dataset
if not val_exists:
    print(f"\nGenerating {N_VAL_SAMPLES:,} validation images...")
    start = time.time()

    !python scripts/generate_demo_dataset.py \
        --n {N_VAL_SAMPLES} \
        --format {IMAGE_FORMAT} \
        --difficulty mixed \
        --realism medium \
        --seed 99999 \
        --output-dir {VAL_DATA_DIR}

    elapsed = time.time() - start
    print(f"Validation generation complete in {elapsed:.0f}s")
else:
    print("Validation dataset already exists, skipping generation.")

In [ ]:
# Verify dataset integrity
from pathlib import Path
import csv

total_images = 0
synth_path = Path(SYNTH_DATA_DIR)

shard_dirs = sorted(synth_path.glob('shard_*'))
if shard_dirs:
    for shard in shard_dirs:
        csv_path = shard / 'labels.csv'
        if csv_path.exists():
            with open(csv_path) as f:
                n = sum(1 for _ in csv.reader(f)) - 1
            imgs = len(list((shard / 'images').glob('*'))) if (shard / 'images').exists() else 0
            total_images += n
            print(f"  {shard.name}: {n:,} labels, {imgs:,} images")
else:
    csv_path = synth_path / 'labels.csv'
    if csv_path.exists():
        with open(csv_path) as f:
            total_images = sum(1 for _ in csv.reader(f)) - 1

print(f"\nTotal training images: {total_images:,}")

# Check size on disk
import subprocess
result = subprocess.run(['du', '-sh', SYNTH_DATA_DIR], capture_output=True, text=True)
print(f"Disk usage: {result.stdout.strip()}")

if Path(VAL_DATA_DIR).exists():
    result = subprocess.run(['du', '-sh', VAL_DATA_DIR], capture_output=True, text=True)
    print(f"Validation: {result.stdout.strip()}")

---
## 4. Phase 1: Synthetic-Only Training

Train on 200k synthetic images. If `P1_MODE='compare'`, runs both Ogham Unicode
and Latin transliteration modes and produces a comparison report.

**Time estimate:** ~5 hours for 20 epochs with `trocr-small-stage1` on A100.

Checkpoints are saved to Drive after every improvement, so training survives
Colab disconnects. Use `--resume` to continue from the best checkpoint.

In [ ]:
# Phase 1: Synthetic-only training
!python scripts/train_colab.py \
    --mode {P1_MODE} \
    --model-size {MODEL_SIZE} \
    --phase 1 \
    --data-dir {SYNTH_DATA_DIR} \
    --val-data-dir {VAL_DATA_DIR} \
    --font-dir {FONT_DIR_STR} \
    --epochs {P1_EPOCHS} \
    --batch-size {P1_BATCH_SIZE} \
    --lr {P1_LR} \
    --freeze-encoder-epochs {P1_FREEZE_ENCODER} \
    --init-strategy {INIT_STRATEGY} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --num-workers 4

In [ ]:
# Resume Phase 1 after disconnect (safe to re-run)
# Uncomment and run this cell if your session was interrupted:

# !python scripts/train_colab.py \
#     --mode {P1_MODE} \
#     --model-size {MODEL_SIZE} \
#     --phase 1 \
#     --data-dir {SYNTH_DATA_DIR} \
#     --val-data-dir {VAL_DATA_DIR} \
#     --font-dir {FONT_DIR_STR} \
#     --epochs {P1_EPOCHS} \
#     --batch-size {P1_BATCH_SIZE} \
#     --lr {P1_LR} \
#     --freeze-encoder-epochs {P1_FREEZE_ENCODER} \
#     --init-strategy {INIT_STRATEGY} \
#     --checkpoint-dir {CHECKPOINT_DIR} \
#     --resume \
#     --num-workers 4

---
## 5. Phase 1 Evaluation

Inspect training history, loss curves, and sample predictions.

In [ ]:
# Load and display training history
import json
from pathlib import Path

def load_history(mode):
    path = Path(CHECKPOINT_DIR) / f'history_{mode}.json'
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# Check which modes were trained
for mode in ['ogham', 'latin']:
    h = load_history(mode)
    if h:
        best_cer = min(h['val_cer'])
        best_epoch = h['val_cer'].index(best_cer) + 1
        final_loss = h['train_loss'][-1]
        print(f"{mode.upper()} mode:")
        print(f"  Best CER: {best_cer:.4f} (epoch {best_epoch})")
        print(f"  Final train loss: {final_loss:.4f}")
        print(f"  Final exact match: {h['val_exact_match'][-1]:.1%}")
        print()

# Load comparison report if available
report_path = Path(CHECKPOINT_DIR) / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print("COMPARISON REPORT:")
    for mode, r in report.items():
        print(f"  {mode}: Best CER = {r['best_cer']:.4f}")
    winner = min(report, key=lambda m: report[m]['best_cer'])
    print(f"\n  Winner: {winner.upper()}")

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'ogham': '#2196F3', 'latin': '#FF9800'}

for mode in ['ogham', 'latin']:
    h = load_history(mode)
    if h is None:
        continue
    epochs = range(1, len(h['train_loss']) + 1)
    c = colors[mode]

    # Train & val loss
    axes[0].plot(epochs, h['train_loss'], '-', color=c, alpha=0.6, label=f'{mode} train')
    axes[0].plot(epochs, h['val_loss'], '--', color=c, label=f'{mode} val')

    # CER
    axes[1].plot(epochs, h['val_cer'], '-', color=c, label=mode)

    # Exact match
    axes[2].plot(epochs, [x * 100 for x in h['val_exact_match']], '-', color=c, label=mode)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend()

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('CER')
axes[1].set_title('Character Error Rate (lower is better)'); axes[1].legend()

axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Exact Match %')
axes[2].set_title('Exact Match Rate'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/phase1_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {CHECKPOINT_DIR}/phase1_curves.png")

In [ ]:
# Sample predictions from best checkpoint
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import csv, random

# Pick the winning mode (or default to ogham)
report_path = Path(CHECKPOINT_DIR) / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    best_mode = min(report, key=lambda m: report[m]['best_cer'])
else:
    best_mode = 'ogham'

ckpt_path = Path(CHECKPOINT_DIR) / f'best_{best_mode}'
print(f"Loading best checkpoint: {ckpt_path} (mode: {best_mode})")

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-stage1')
model = VisionEncoderDecoderModel.from_pretrained(str(ckpt_path))
tokenizer_path = ckpt_path
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Load some validation samples
val_path = Path(VAL_DATA_DIR)
csv_path = val_path / 'labels.csv'
if not csv_path.exists():
    # Try shard layout
    shard_dirs = sorted(val_path.glob('shard_*'))
    csv_path = shard_dirs[0] / 'labels.csv' if shard_dirs else csv_path

with open(csv_path) as f:
    reader = list(csv.DictReader(f))

samples = random.sample(reader, min(10, len(reader)))

print(f"\nSample predictions ({best_mode} mode):")
print(f"{'Reference':<30} {'Prediction':<30} {'Match'}")
print('-' * 70)

for row in samples:
    img_path = val_path / row['image_file']
    if not img_path.exists():
        img_path = val_path / 'images' / row['image_file']
    if not img_path.exists():
        continue

    image = Image.open(img_path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_length=64, num_beams=4)
    pred = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    ref_key = 'ogham_text' if best_mode == 'ogham' else 'latin_transliteration'
    ref = row[ref_key]
    match = 'Y' if pred.strip() == ref.strip() else ''
    print(f"{ref:<30} {pred:<30} {match}")

---
## 6. Phase 2: Curriculum Learning with Real Data

Fine-tune the Phase 1 winner using real stone images mixed with synthetic data.
Curriculum schedule: 95% synthetic (epoch 0) -> 50/50 (epoch 50).

**Prerequisites:**
- Real stone data uploaded to `{REAL_DATA_DIR}` with `splits/` and `processed/` directories
- Phase 1 checkpoint available

**Time estimate:** ~8 hours for 50 epochs on A100.

In [ ]:
# Check real data availability
real_path = Path(REAL_DATA_DIR)
split_file = real_path / 'splits' / 'train_stones.txt'

if split_file.exists():
    with open(split_file) as f:
        n_train = sum(1 for line in f if line.strip())
    with open(real_path / 'splits' / 'val_stones.txt') as f:
        n_val = sum(1 for line in f if line.strip())
    with open(real_path / 'splits' / 'test_stones.txt') as f:
        n_test = sum(1 for line in f if line.strip())
    print(f"Real data found: {n_train} train / {n_val} val / {n_test} test stones")
    print("Ready for Phase 2!")
else:
    print("Real data not found.")
    print(f"Upload your ogham_dataset/ directory to: {real_path}")
    print("Required structure:")
    print(f"  {real_path}/splits/train_stones.txt")
    print(f"  {real_path}/splits/val_stones.txt")
    print(f"  {real_path}/splits/test_stones.txt")
    print(f"  {real_path}/processed/annotations/transcriptions.json")
    print(f"  {real_path}/processed/cropped/<stone_id>/*.png")

In [ ]:
# Determine winning mode from Phase 1
report_path = Path(CHECKPOINT_DIR) / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    WINNING_MODE = min(report, key=lambda m: report[m]['best_cer'])
    print(f"Phase 1 winner: {WINNING_MODE} (CER: {report[WINNING_MODE]['best_cer']:.4f})")
else:
    WINNING_MODE = 'ogham'  # Default
    print(f"No comparison report found, defaulting to: {WINNING_MODE}")

print(f"\nPhase 2 will train {WINNING_MODE} mode with curriculum learning.")

In [ ]:
# Phase 2: Curriculum training with real + synthetic data
!python scripts/train_colab.py \
    --mode {WINNING_MODE} \
    --model-size {MODEL_SIZE} \
    --phase 2 \
    --data-dir {SYNTH_DATA_DIR} \
    --val-data-dir {VAL_DATA_DIR} \
    --real-data-dir {REAL_DATA_DIR} \
    --font-dir {FONT_DIR_STR} \
    --epochs {P2_EPOCHS} \
    --batch-size {P2_BATCH_SIZE} \
    --lr {P2_LR} \
    --freeze-encoder-epochs 0 \
    --init-strategy {INIT_STRATEGY} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --resume \
    --num-workers 4

---
## 7. Final Evaluation

Evaluate the final model on the held-out real test set (21 stones).

In [ ]:
# Load final model and evaluate on real test set
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer
from pathlib import Path
import json

# Determine best mode
report_path = Path(CHECKPOINT_DIR) / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    best_mode = min(report, key=lambda m: report[m]['best_cer'])
else:
    best_mode = 'ogham'

ckpt = Path(CHECKPOINT_DIR) / f'best_{best_mode}'
print(f"Evaluating: {ckpt} (mode: {best_mode})")

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-stage1')
model = VisionEncoderDecoderModel.from_pretrained(str(ckpt))
tokenizer = AutoTokenizer.from_pretrained(str(ckpt))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()

# Load real test data
real_path = Path(REAL_DATA_DIR)
test_file = real_path / 'splits' / 'test_stones.txt'

if not test_file.exists():
    print("No real test data available. Skipping real evaluation.")
else:
    with open(test_file) as f:
        test_stones = [line.strip() for line in f if line.strip()]

    with open(real_path / 'processed' / 'annotations' / 'transcriptions.json') as f:
        annotations = json.load(f)

    from PIL import Image
    import sys
    sys.path.insert(0, REPO_DIR)

    all_preds, all_refs = [], []

    for stone_id in test_stones:
        if stone_id not in annotations:
            continue
        ann = annotations[stone_id]
        ref = ann.get('transcription', '')
        if not ref:
            continue

        # Convert to latin if needed
        if best_mode == 'latin':
            from src.utils.ogham import OGHAM_TO_LATIN
            ref = ''.join(OGHAM_TO_LATIN.get(ch, ch) for ch in ref)

        # Find images
        crop_dir = real_path / 'processed' / 'cropped' / stone_id
        if not crop_dir.exists():
            continue
        image_paths = list(crop_dir.glob('*.png')) + list(crop_dir.glob('*.jpg'))

        for img_path in image_paths:
            image = Image.open(img_path).convert('RGB')
            pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)

            with torch.no_grad():
                gen_ids = model.generate(pixel_values, max_length=64, num_beams=4)
            pred = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

            all_preds.append(pred)
            all_refs.append(ref)

    # Compute metrics
    from scripts.train_colab import compute_cer
    cer = compute_cer(all_preds, all_refs)
    exact = sum(1 for p, r in zip(all_preds, all_refs) if p.strip() == r.strip()) / max(len(all_preds), 1)

    print(f"\n{'='*60}")
    print(f"FINAL TEST RESULTS ({best_mode} mode)")
    print(f"{'='*60}")
    print(f"  Stones evaluated: {len(test_stones)}")
    print(f"  Total images: {len(all_preds)}")
    print(f"  Character Error Rate: {cer:.4f}")
    print(f"  Exact Match: {exact:.1%}")
    print(f"{'='*60}")

    # Show per-stone results
    print(f"\nSample predictions:")
    for i in range(min(15, len(all_preds))):
        match = 'Y' if all_preds[i].strip() == all_refs[i].strip() else ''
        print(f"  ref='{all_refs[i]}'  pred='{all_preds[i]}'  {match}")

    # Save results
    results = {'mode': best_mode, 'cer': cer, 'exact_match': exact,
               'n_stones': len(test_stones), 'n_images': len(all_preds)}
    with open(Path(CHECKPOINT_DIR) / 'test_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {CHECKPOINT_DIR}/test_results.json")

---
## 8. Export Model

Export the best model for deployment or sharing.

In [ ]:
# Export best model to a clean directory on Drive
import shutil
from pathlib import Path

# Determine best mode
report_path = Path(CHECKPOINT_DIR) / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    best_mode = min(report, key=lambda m: report[m]['best_cer'])
else:
    best_mode = 'ogham'

src_ckpt = Path(CHECKPOINT_DIR) / f'best_{best_mode}'
export_dir = Path(f'{DRIVE_ROOT}/exports/ogham_ocr_{best_mode}_{MODEL_SIZE}')
export_dir.mkdir(parents=True, exist_ok=True)

# Copy model files
for f in src_ckpt.iterdir():
    shutil.copy2(f, export_dir / f.name)

# Copy training curves and results
for extra in ['phase1_curves.png', 'comparison_report.json', 'test_results.json',
              f'history_{best_mode}.json']:
    src = Path(CHECKPOINT_DIR) / extra
    if src.exists():
        shutil.copy2(src, export_dir / extra)

# Create model card
card = f"""# Ogham OCR Model

Fine-tuned TrOCR for ancient Ogham inscription recognition.

- **Base model**: microsoft/trocr-{MODEL_SIZE}-stage1
- **Output mode**: {best_mode}
- **Init strategy**: {INIT_STRATEGY}
- **Training data**: {N_TRAIN_SAMPLES:,} synthetic images

## Usage

```python
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer
from PIL import Image

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-{MODEL_SIZE}-stage1')
model = VisionEncoderDecoderModel.from_pretrained('{export_dir}')
tokenizer = AutoTokenizer.from_pretrained('{export_dir}')

image = Image.open('ogham_inscription.jpg').convert('RGB')
pixel_values = processor(images=image, return_tensors='pt').pixel_values

generated_ids = model.generate(pixel_values, max_length=64, num_beams=4)
text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(text)
```
"""

with open(export_dir / 'README.md', 'w') as f:
    f.write(card)

# Show what was exported
print(f"Model exported to: {export_dir}")
print("\nContents:")
for f in sorted(export_dir.iterdir()):
    size = f.stat().st_size
    if size > 1e6:
        print(f"  {f.name} ({size/1e6:.1f} MB)")
    else:
        print(f"  {f.name} ({size/1e3:.0f} KB)")